# Generate Numbered Sync-Check Timing Sweep Commands

This notebook builds repeatable commands for `run_camera_sync_check.sh`. The sync-check workflow is the right timing diagnostic because DMD A shows a known number sequence and DMD B holds a small dot. Each trigger/frame can then be compared against the expected number.

Defaults here use numbers on A at `100 px` and a B dot radius of `20 px`.

In [ ]:
from pathlib import Path
import csv
import itertools
import shlex
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
OUTPUT_ROOT = Path("runs/camera")
SWEEP_ID = datetime.now().strftime("sync-timing-sweep-%Y%m%d-%H%M%S")
SWEEP_MANIFEST_PATH = REPO_ROOT / "runs" / f"{SWEEP_ID}_manifest.csv"
SWEEP_COMMAND_PATH = REPO_ROOT / "runs" / f"{SWEEP_ID}_command.sh"

REPO_ROOT, SWEEP_MANIFEST_PATH, SWEEP_COMMAND_PATH

## Sweep Parameters

Keep the first sweep small. At 60 Hz, the requested numbers are packed into one LUT cycle, so `len(numbers) * (exposure_us + dark_time_us)` must fit inside the usable frame budget. The defaults below stay conservative.

In [ ]:
numbers = [1, 2, 3, 4, 5]
number_size_px = 100
b_dot_radius = 20
b_dot_x = 960
b_dot_y = 540

exposure_us_values = [600, 1000, 1500, 2000]
dark_time_us_values = [0, 100, 250, 500]
trigger_rising_delay_us_values = [0, -20, 20, 50]
repeats_per_setting = 1

base_capture = {
    "test": "a-numbers-b-static",
    "test_b": "dot",
    "numbers": ",".join(str(n) for n in numbers),
    "number_size_px": number_size_px,
    "b_dot_x": b_dot_x,
    "b_dot_y": b_dot_y,
    "b_dot_radius": b_dot_radius,
    "runtime_seconds": 0,
    "polarity_mode": "ignore",
    "event_noise_filter": "none",
    "save_filtered_events": True,
    "output_root": str(OUTPUT_ROOT),
    "verbose": 1,
}

len(exposure_us_values) * len(dark_time_us_values) * len(trigger_rising_delay_us_values) * repeats_per_setting

In [ ]:
def rising_delay_tag(value):
    return str(int(value)).replace("-", "m")


def build_sync_check_argv(row):
    cmd = [
        "--test", row["test"],
        "--test-b", row["test_b"],
        "--numbers", row["numbers"],
        "--number-size-px", str(row["number_size_px"]),
        "--b-dot-x", str(row["b_dot_x"]),
        "--b-dot-y", str(row["b_dot_y"]),
        "--b-dot-radius", str(row["b_dot_radius"]),
        "--exposure-us", str(row["exposure_us"]),
        "--dark-time-us", str(row["dark_time_us"]),
        "--trigger-out-2-rising-delay-us", str(row["trigger_rising_delay_us"]),
        "--runtime-seconds", str(row["runtime_seconds"]),
        "--polarity-mode", row["polarity_mode"],
        "--event-noise-filter", row["event_noise_filter"],
        "--output-root", row["output_root"],
        "--name-override", row["timestamp"],
    ]
    if row["save_filtered_events"]:
        cmd.append("--save-filtered-events")
    if row["verbose"]:
        cmd.extend(["-" + "v" * int(row["verbose"])])
    return cmd


rows = []
index = 0
for repeat, exposure_us, dark_time_us, trigger_rising_delay_us in itertools.product(
    range(repeats_per_setting),
    exposure_us_values,
    dark_time_us_values,
    trigger_rising_delay_us_values,
):
    timestamp = (
        f"{SWEEP_ID}-{index:03d}"
        f"-exp{exposure_us}us"
        f"-dark{dark_time_us}us"
        f"-rise{rising_delay_tag(trigger_rising_delay_us)}us"
        f"-r{repeat}"
    )
    row = dict(base_capture)
    row.update(
        index=index,
        sweep_id=SWEEP_ID,
        repeat=repeat,
        timestamp=timestamp,
        exposure_us=exposure_us,
        dark_time_us=dark_time_us,
        trigger_rising_delay_us=trigger_rising_delay_us,
    )
    row["sync_check_argv"] = shlex.join(build_sync_check_argv(row))
    rows.append(row)
    index += 1

rows[:3]

In [ ]:
SWEEP_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
# sync-sweep consumes sync_check_argv or command. The other columns are metadata for analysis.
fieldnames = list(rows[0].keys())
with SWEEP_MANIFEST_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

sweep_command = ["./run_camera_sync_sweep.sh", "--manifest", str(SWEEP_MANIFEST_PATH)]

with SWEEP_COMMAND_PATH.open("w", encoding="utf-8") as handle:
    handle.write("#!/usr/bin/env bash\n")
    handle.write("set -euo pipefail\n\n")
    handle.write(shlex.join(sweep_command) + "\n")

print(f"Wrote {len(rows)} sync-check manifest rows")
print(SWEEP_MANIFEST_PATH)
print(SWEEP_COMMAND_PATH)

## Run Plan

Run the generated shell file on the DMD machine from the repo root. It launches one persistent `camera sync-sweep` process, opens the camera once, and reuses that handle for every timing row in the manifest. Each manifest row carries an explicit `sync_check_argv`; `sync-sweep` does not infer command flags from separate CSV option columns.

In [ ]:
print("Persistent sweep command:")
print(shlex.join(sweep_command))

print("\nFirst manifest rows:")
for row in rows[:10]:
    print(row["sync_check_argv"])
    print()